# Day 4 Practical: Exploring Protein Structure with AlphaFold
**AI for Drug Discovery**

## Overview and Learning Objectives

In this practical, you will explore **AlphaFold** — DeepMind's revolutionary AI system that predicts protein 3D structures from amino acid sequences with near-experimental accuracy. By the end of this notebook, you will be able to:

1. **Understand the protein folding problem** — why predicting 3D structure from sequence was one of biology's grand challenges
2. **Access the AlphaFold Protein Structure Database** — retrieve predicted structures for any protein using its UniProt ID
3. **Visualize protein structures** in 3D and analyze confidence scores (pLDDT)
4. **Identify drug-binding sites** by examining structural features of pharmacological targets
5. **Compare AlphaFold predictions with experimental structures** from the Protein Data Bank (PDB)
6. **Understand the implications for drug discovery** — how AI-predicted structures accelerate the drug design pipeline

### Why Protein Structure Matters for Drug Discovery

Most drugs work by binding to specific proteins in the body. To design a drug that binds tightly and selectively to its target, we need to know the 3D shape of that target — the arrangement of atoms in space that forms the binding pocket. Before AlphaFold, obtaining a protein structure required expensive and time-consuming experimental techniques (X-ray crystallography, cryo-EM, NMR). AlphaFold changed this by providing predicted structures for over 200 million proteins — essentially every known protein sequence.

### What is AlphaFold?

AlphaFold is a deep learning system developed by **DeepMind** (a subsidiary of Google/Alphabet) that predicts the 3D structure of a protein from its amino acid sequence. It was first demonstrated at **CASP13 (2018)** and achieved breakthrough accuracy at **CASP14 (2020)**, solving the protein folding problem for single-domain proteins. The system uses:

- **Multiple sequence alignments (MSAs)**: Evolutionary information from related proteins
- **Pair representations**: Relationships between all pairs of residues
- **Evoformer blocks**: A novel neural network architecture combining attention mechanisms
- **Structure module**: Iteratively refines 3D coordinates

Key publications:
- Jumper et al. (2021). "Highly accurate protein structure prediction with AlphaFold." *Nature*, 596:583-589.
- Varadi et al. (2022). "AlphaFold Protein Structure Database: massively expanding structural coverage." *Nucleic Acids Research*, 50:D439-D444.

### Connection to Drug Discovery

AlphaFold structures are now routinely used in:
- **Structure-based drug design**: Docking small molecules into predicted binding sites
- **Target identification**: Understanding which proteins are "druggable"
- **Selectivity optimization**: Designing drugs that bind one target but not close relatives
- **Antibody design**: Predicting antibody-antigen interactions
- **Understanding disease mechanisms**: Visualizing how mutations affect protein structure

In [ ]:
# ============================================================
# INSTALL REQUIRED PYTHON PACKAGES
# ============================================================
# We need several packages for this practical:
#   - requests: HTTP library for accessing the AlphaFold Database API
#   - numpy: numerical computing for array operations
#   - matplotlib: plotting library for visualizations
#   - pandas: data manipulation with DataFrames
#   - py3Dmol: molecular visualization in Jupyter notebooks
#   - biopython: biological computation library for parsing PDB files
!pip install requests numpy matplotlib pandas py3Dmol biopython -q

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# IMPORT ALL REQUIRED LIBRARIES
# ============================================================

# requests: for making HTTP requests to the AlphaFold Database API
import requests

# numpy: numerical operations on arrays
import numpy as np

# pandas: tabular data manipulation
import pandas as pd

# matplotlib: plotting
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# py3Dmol: interactive 3D molecular visualization
import py3Dmol

# BioPython PDB parser for structural analysis
from Bio.PDB import PDBParser, DSSP, Select
from Bio.PDB.Polypeptide import protein_letters_3to1
import io
import json

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

print('All imports successful!')

## Part 1: The Protein Folding Problem

A protein's function is determined by its 3D structure. The sequence of amino acids (the primary structure) folds into a specific 3D shape through:

- **Secondary structure**: Local folding patterns (α-helices, β-sheets)
- **Tertiary structure**: Overall 3D arrangement of a single chain
- **Quaternary structure**: Assembly of multiple chains

The **protein folding problem** asks: given only the amino acid sequence, can we predict the final 3D structure? This was considered one of the grand challenges in biology for 50+ years. Levinthal's paradox (1969) noted that a protein with 100 residues has ~3^198 possible conformations — more than the atoms in the universe — yet proteins fold in milliseconds.

### The pLDDT Confidence Score

AlphaFold provides a per-residue confidence score called **pLDDT** (predicted Local Distance Difference Test), ranging from 0 to 100:

| pLDDT Range | Interpretation | Typical Regions |
|---|---|---|
| > 90 | Very high confidence | Well-structured cores |
| 70–90 | Confident | Most of the protein |
| 50–70 | Low confidence | Flexible loops, domain boundaries |
| < 50 | Very low confidence | Disordered regions, termini |

In [ ]:
# ============================================================
# ACCESSING THE ALPHAFOLD PROTEIN STRUCTURE DATABASE
# ============================================================
# The AlphaFold Database (https://alphafold.com/) provides
# predicted structures for over 200 million proteins.
# We access it via a REST API using UniProt accession IDs.
#
# NOTE: The database migrated from alphafold.ebi.ac.uk to
# alphafold.com in 2025. We use both domains as fallback.

# Base URLs for the AlphaFold Database API (new and legacy)
ALPHAFOLD_API_URLS = [
    'https://alphafold.com/api',
    'https://alphafold.ebi.ac.uk/api',
]

# Base URLs for PDB file downloads
ALPHAFOLD_FILES_URLS = [
    'https://alphafold.com/files',
    'https://alphafold.ebi.ac.uk/files',
]

def get_alphafold_prediction(uniprot_id):
    """
    Retrieve AlphaFold prediction metadata for a given UniProt ID.

    Parameters:
        uniprot_id (str): UniProt accession (e.g., 'P00533' for EGFR)

    Returns:
        dict: Prediction metadata including URLs for structure files
    """
    for base_url in ALPHAFOLD_API_URLS:
        url = f'{base_url}/prediction/{uniprot_id}'
        try:
            response = requests.get(url, timeout=30)
            if response.status_code == 200:
                return response.json()[0]  # Returns a list; take first entry
        except requests.exceptions.RequestException:
            continue

    print(f'Error: Could not retrieve prediction for {uniprot_id} from any API endpoint')
    return None

def download_pdb(uniprot_id):
    """
    Download the AlphaFold predicted PDB file for a given UniProt ID.
    Tries multiple file versions (v4, v3, v2) and both old/new domains.

    Parameters:
        uniprot_id (str): UniProt accession

    Returns:
        str: PDB file contents as a string
    """
    # Try different model versions (v4 is latest, fall back to v3/v2)
    versions = ['v4', 'v3', 'v2']

    for base_url in ALPHAFOLD_FILES_URLS:
        for version in versions:
            url = f'{base_url}/AF-{uniprot_id}-F1-model_{version}.pdb'
            try:
                response = requests.get(url, timeout=30)
                if response.status_code == 200:
                    return response.text
            except requests.exceptions.RequestException:
                continue

    print(f'Error: Could not download PDB for {uniprot_id} (tried all versions and endpoints)')
    return None

# ============================================================
# TEST: Retrieve metadata for human Acetylcholinesterase (AChE)
# ============================================================
# AChE is the enzyme that breaks down the neurotransmitter acetylcholine.
# It is the target of nerve agents (sarin, VX) and Alzheimer's drugs
# (donepezil, rivastigmine). UniProt ID: P22303

ache_uniprot = 'P22303'

try:
    ache_info = get_alphafold_prediction(ache_uniprot)
except Exception as e:
    print(f"⚠️  Network error: {e}")
    print("This notebook requires internet access to the AlphaFold Database.")
    print("Please run this notebook in Google Colab or a local Jupyter environment.")
    ache_info = None

if ache_info:
    print(f"Protein: {ache_info.get('uniprotDescription', 'N/A')}")
    print(f"Organism: {ache_info.get('organismScientificName', 'N/A')}")
    print(f"UniProt ID: {ache_info.get('uniprotAccession', 'N/A')}")
    print(f"Sequence length: {ache_info.get('uniprotEnd', ache_info.get('sequenceEnd', 'N/A'))} residues")
    print(f"Model URL: {ache_info.get('pdbUrl', 'N/A')}")
    print(f"\nGlobal pLDDT (mean confidence): check per-residue below")


In [ ]:
# ============================================================
# VISUALIZE THE 3D STRUCTURE WITH py3Dmol
# ============================================================
# We'll download and visualize the AlphaFold predicted structure
# of Acetylcholinesterase, colored by pLDDT confidence.

# Download the PDB file
ache_pdb = download_pdb(ache_uniprot)

if ache_pdb:
    # Create a 3D viewer
    view = py3Dmol.view(width=800, height=500)
    
    # Add the protein structure
    view.addModel(ache_pdb, 'pdb')
    
    # Color by B-factor (which AlphaFold uses to store pLDDT scores)
    # Blue = high confidence (>90), Red = low confidence (<50)
    view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'rwb', 'min': 50, 'max': 90}}})
    
    # Zoom to fit
    view.zoomTo()
    
    print("Acetylcholinesterase (AChE) - AlphaFold Predicted Structure")
    print("Color scheme: Blue = high confidence (pLDDT > 90), Red = low confidence (pLDDT < 50)")
    print("White = intermediate confidence (pLDDT 50-90)")
    view.show()

In [ ]:
# ============================================================
# EXTRACT AND ANALYZE pLDDT SCORES
# ============================================================
# The B-factor column in AlphaFold PDB files contains the pLDDT
# confidence score for each residue. Let's extract and plot these.

def extract_plddt_from_pdb(pdb_text):
    """
    Extract per-residue pLDDT scores from an AlphaFold PDB file.
    In AlphaFold PDB files, the B-factor column stores pLDDT values.
    We take the Cα (CA) atom B-factor as the residue-level score.
    
    Parameters:
        pdb_text (str): PDB file content as string
    
    Returns:
        tuple: (residue_numbers, plddt_scores, residue_names)
    """
    residue_numbers = []
    plddt_scores = []
    residue_names = []
    
    for line in pdb_text.split('\n'):
        # ATOM records contain coordinate data
        # We only look at CA (alpha carbon) atoms — one per residue
        if line.startswith('ATOM') and line[12:16].strip() == 'CA':
            res_num = int(line[22:26].strip())
            b_factor = float(line[60:66].strip())  # pLDDT is stored here
            res_name = line[17:20].strip()
            
            residue_numbers.append(res_num)
            plddt_scores.append(b_factor)
            residue_names.append(res_name)
    
    return np.array(residue_numbers), np.array(plddt_scores), residue_names

# Extract pLDDT scores from our AChE structure
res_nums, plddt, res_names = extract_plddt_from_pdb(ache_pdb)

print(f'Number of residues: {len(plddt)}')
print(f'Mean pLDDT: {plddt.mean():.1f}')
print(f'Median pLDDT: {np.median(plddt):.1f}')
print(f'Min pLDDT: {plddt.min():.1f} (residue {res_nums[plddt.argmin()]})')
print(f'Max pLDDT: {plddt.max():.1f} (residue {res_nums[plddt.argmax()]})')
print(f'\nResidues with very high confidence (>90): {(plddt > 90).sum()} ({100*(plddt > 90).mean():.1f}%)')
print(f'Residues with low confidence (<70): {(plddt < 70).sum()} ({100*(plddt < 70).mean():.1f}%)')
print(f'Residues with very low confidence (<50): {(plddt < 50).sum()} ({100*(plddt < 50).mean():.1f}%)')

In [ ]:
# ============================================================
# PLOT pLDDT SCORES ALONG THE PROTEIN SEQUENCE
# ============================================================
# This is a standard visualization in structural biology.
# It shows which parts of the protein are well-predicted (structured)
# and which are uncertain (disordered/flexible).

fig, ax = plt.subplots(figsize=(14, 4))

# Color regions by confidence level
ax.fill_between(res_nums, plddt, alpha=0.3, color='steelblue')
ax.plot(res_nums, plddt, linewidth=0.8, color='steelblue')

# Add horizontal reference lines for confidence thresholds
ax.axhline(y=90, color='green', linestyle='--', alpha=0.7, label='Very high (>90)')
ax.axhline(y=70, color='orange', linestyle='--', alpha=0.7, label='Confident (>70)')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.7, label='Low confidence (>50)')

ax.set_xlabel('Residue Number', fontsize=12)
ax.set_ylabel('pLDDT Score', fontsize=12)
ax.set_title('AlphaFold Confidence (pLDDT) for Human Acetylcholinesterase', fontsize=14)
ax.set_ylim(0, 100)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- High pLDDT regions are well-structured (helices, sheets, stable loops)")
print("- Low pLDDT regions are likely flexible or disordered (N/C termini, loops)")
print("- The catalytic core of AChE is highly structured → high pLDDT")
print("- Signal peptides and flexible termini → low pLDDT")

## Part 2: Comparing Drug Targets — Multiple Proteins

Let's examine several important drug targets to understand how AlphaFold predictions vary across different protein families. We'll look at:

1. **Cyclooxygenase-2 (COX-2)** — Target of NSAIDs like ibuprofen and celecoxib
2. **Epidermal Growth Factor Receptor (EGFR)** — Target of cancer drugs like gefitinib
3. **GABA-A receptor α1 subunit** — Target of benzodiazepines (e.g., diazepam)
4. **Dopamine D2 receptor** — Target of antipsychotic drugs

In [ ]:
# ============================================================
# COMPARE MULTIPLE DRUG TARGETS
# ============================================================
# Define important drug targets with their UniProt IDs and descriptions

drug_targets = {
    'P35354': {'name': 'Cyclooxygenase-2 (COX-2)', 'drug': 'Celecoxib (Celebrex)', 'class': 'Enzyme'},
    'P00533': {'name': 'EGFR (ErbB1)', 'drug': 'Gefitinib (Iressa)', 'class': 'Kinase'},
    'P14867': {'name': 'GABA-A receptor α1', 'drug': 'Diazepam (Valium)', 'class': 'Ion channel'},
    'P14416': {'name': 'Dopamine D2 receptor', 'drug': 'Haloperidol', 'class': 'GPCR'},
}

# Download and analyze each target
target_data = {}

for uniprot_id, info in drug_targets.items():
    print(f"\nDownloading: {info['name']} ({uniprot_id})...")
    pdb_text = download_pdb(uniprot_id)
    
    if pdb_text:
        res_nums, plddt, res_names = extract_plddt_from_pdb(pdb_text)
        target_data[uniprot_id] = {
            'plddt': plddt,
            'res_nums': res_nums,
            'pdb': pdb_text,
            'info': info
        }
        print(f"  → {len(plddt)} residues, mean pLDDT = {plddt.mean():.1f}")
    else:
        print(f"  → Failed to download")

print(f"\nSuccessfully loaded {len(target_data)} drug targets.")

In [ ]:
# ============================================================
# COMPARE pLDDT DISTRIBUTIONS ACROSS TARGETS
# ============================================================
# Different protein families have different structural characteristics.
# GPCRs and ion channels (membrane proteins) often have lower confidence
# in their transmembrane regions compared to soluble enzymes.

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, (uniprot_id, data) in enumerate(target_data.items()):
    ax = axes[idx]
    info = data['info']
    plddt = data['plddt']
    res_nums = data['res_nums']
    
    # Plot pLDDT along sequence
    ax.fill_between(res_nums, plddt, alpha=0.3, color='steelblue')
    ax.plot(res_nums, plddt, linewidth=0.8, color='steelblue')
    ax.axhline(y=70, color='orange', linestyle='--', alpha=0.5)
    ax.axhline(y=90, color='green', linestyle='--', alpha=0.5)
    
    ax.set_ylim(0, 100)
    ax.set_xlabel('Residue', fontsize=10)
    ax.set_ylabel('pLDDT', fontsize=10)
    ax.set_title(f"{info['name']}\nDrug: {info['drug']} | Mean pLDDT: {plddt.mean():.1f}", fontsize=10)

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "="*80)
print(f"{'Protein':<30} {'Class':<12} {'Residues':<10} {'Mean pLDDT':<12} {'% > 90':<10}")
print("="*80)
for uniprot_id, data in target_data.items():
    info = data['info']
    plddt = data['plddt']
    pct_high = 100 * (plddt > 90).mean()
    print(f"{info['name']:<30} {info['class']:<12} {len(plddt):<10} {plddt.mean():<12.1f} {pct_high:<10.1f}")

In [ ]:
# ============================================================
# VISUALIZE A DRUG TARGET IN 3D: EGFR Kinase Domain
# ============================================================
# EGFR is a receptor tyrosine kinase frequently mutated in cancers.
# Drugs like gefitinib and erlotinib bind the ATP-binding site in
# the kinase domain. Let's visualize the AlphaFold prediction.

egfr_data = target_data.get('P00533')

if egfr_data:
    view = py3Dmol.view(width=800, height=500)
    view.addModel(egfr_data['pdb'], 'pdb')
    
    # Show as cartoon colored by pLDDT (B-factor)
    view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'rwb', 'min': 50, 'max': 90}}})
    view.zoomTo()
    
    print("EGFR (Epidermal Growth Factor Receptor) — AlphaFold Prediction")
    print("Target of cancer drugs: gefitinib, erlotinib, osimertinib")
    print("Color: Blue = high confidence, Red = low confidence")
    view.show()

## Part 3: Structural Analysis — Finding Drug Binding Sites

For drug discovery, we're most interested in the **binding site** — the region of the protein where a drug molecule can bind. Binding sites typically:

- Are **concave pockets** on the protein surface
- Have **hydrophobic character** (to accommodate drug-like molecules)
- Are **well-structured** (high pLDDT in AlphaFold predictions)
- Are **conserved** across species (evolutionary importance)

Let's analyze the structural features that make a protein "druggable".

In [ ]:
# ============================================================
# AMINO ACID COMPOSITION AND PROPERTIES
# ============================================================
# Different amino acids have different properties that affect
# drug binding. Let's analyze the composition of our targets.

# Define amino acid properties
aa_properties = {
    'hydrophobic': ['ALA', 'VAL', 'LEU', 'ILE', 'MET', 'PHE', 'TRP', 'PRO'],
    'polar': ['SER', 'THR', 'ASN', 'GLN', 'TYR', 'CYS'],
    'positive': ['LYS', 'ARG', 'HIS'],
    'negative': ['ASP', 'GLU'],
    'special': ['GLY']
}

def analyze_composition(pdb_text):
    """Analyze amino acid composition from PDB text."""
    _, _, res_names = extract_plddt_from_pdb(pdb_text)
    total = len(res_names)
    
    composition = {}
    for category, amino_acids in aa_properties.items():
        count = sum(1 for r in res_names if r in amino_acids)
        composition[category] = 100 * count / total
    
    return composition

# Analyze all targets
print(f"{'Protein':<30} {'Hydrophobic%':<14} {'Polar%':<10} {'Positive%':<12} {'Negative%':<12}")
print("="*78)

compositions = []
for uniprot_id, data in target_data.items():
    comp = analyze_composition(data['pdb'])
    compositions.append(comp)
    info = data['info']
    print(f"{info['name']:<30} {comp['hydrophobic']:<14.1f} {comp['polar']:<10.1f} {comp['positive']:<12.1f} {comp['negative']:<12.1f}")

print("\nNote: Membrane proteins (GPCRs, ion channels) tend to have higher")
print("hydrophobic content due to their transmembrane helices.")

In [ ]:
# ============================================================
# IDENTIFY HIGH-CONFIDENCE STRUCTURED REGIONS
# ============================================================
# For drug design, we focus on well-predicted regions (high pLDDT)
# because these represent reliable structural features.
# Disordered regions (low pLDDT) are unlikely to form stable binding pockets.

def find_structured_regions(plddt_scores, res_nums, threshold=85, min_length=10):
    """
    Find contiguous regions with pLDDT above a threshold.
    These represent well-structured, potentially druggable domains.
    
    Parameters:
        plddt_scores: array of pLDDT values
        res_nums: array of residue numbers
        threshold: minimum pLDDT to consider "structured"
        min_length: minimum number of consecutive residues
    
    Returns:
        list of tuples: (start_residue, end_residue, mean_plddt)
    """
    regions = []
    in_region = False
    start = 0
    
    for i, (score, res) in enumerate(zip(plddt_scores, res_nums)):
        if score >= threshold:
            if not in_region:
                start = i
                in_region = True
        else:
            if in_region:
                length = i - start
                if length >= min_length:
                    mean_plddt = plddt_scores[start:i].mean()
                    regions.append((res_nums[start], res_nums[i-1], mean_plddt, length))
                in_region = False
    
    # Handle case where protein ends in a structured region
    if in_region:
        length = len(plddt_scores) - start
        if length >= min_length:
            mean_plddt = plddt_scores[start:].mean()
            regions.append((res_nums[start], res_nums[-1], mean_plddt, length))
    
    return regions

# Analyze AChE (our neuroscience target)
ache_data = target_data.get('P22303') or {'plddt': plddt, 'res_nums': res_nums}
if 'P22303' not in target_data:
    ache_plddt = plddt
    ache_res_nums = res_nums
else:
    ache_plddt = ache_data['plddt']
    ache_res_nums = ache_data['res_nums']

# Use the pLDDT and res_nums from earlier extraction
ache_res_nums_local, ache_plddt_local, _ = extract_plddt_from_pdb(ache_pdb)
regions = find_structured_regions(ache_plddt_local, ache_res_nums_local)

print("Well-structured regions in Acetylcholinesterase (pLDDT > 85, length > 10):")
print(f"{'Region':<20} {'Residues':<15} {'Length':<10} {'Mean pLDDT':<12}")
print("-"*57)
for i, (start, end, mean_p, length) in enumerate(regions, 1):
    print(f"Region {i:<13} {start}-{end:<11} {length:<10} {mean_p:<12.1f}")

print(f"\nTotal well-structured residues: {sum(r[3] for r in regions)}")
print(f"Percentage of protein: {100*sum(r[3] for r in regions)/len(ache_plddt_local):.1f}%")
print("\nThe catalytic triad of AChE (Ser203, Glu334, His447) sits within")
print("these high-confidence regions — confirming the binding site is well-resolved.")

## Part 4: AlphaFold vs Experimental Structures

How does AlphaFold compare to experimentally determined structures? Let's compare an AlphaFold prediction with the corresponding experimental structure from the **Protein Data Bank (PDB)**.

We'll use the **RCSB PDB** (Research Collaboratory for Structural Bioinformatics) to download experimental structures and compare them to AlphaFold predictions.

In [ ]:
# ============================================================
# DOWNLOAD AN EXPERIMENTAL STRUCTURE FROM PDB
# ============================================================
# PDB ID 4EY7 is the crystal structure of human AChE in complex
# with donepezil (an Alzheimer's drug). Resolution: 2.35 Å

def download_pdb_experimental(pdb_id):
    """
    Download an experimental structure from the RCSB PDB.
    Includes retry logic for transient server errors (e.g., 503).

    Parameters:
        pdb_id (str): 4-character PDB ID (e.g., '4EY7')

    Returns:
        str: PDB file contents
    """
    import time

    # Try multiple URL formats (RCSB sometimes serves from different paths)
    urls = [
        f'https://files.rcsb.org/download/{pdb_id}.pdb',
        f'https://files.rcsb.org/view/{pdb_id}.pdb',
    ]

    for url in urls:
        for attempt in range(3):
            try:
                response = requests.get(url, timeout=30)
                if response.status_code == 200:
                    return response.text
                elif response.status_code == 503:
                    # Server temporarily unavailable — wait and retry
                    print(f'  Server busy (503), retrying in {2**attempt}s...')
                    time.sleep(2**attempt)
                    continue
                else:
                    break  # Try next URL
            except requests.exceptions.RequestException as e:
                print(f'  Connection error: {e}, retrying...')
                time.sleep(2**attempt)
                continue

    print(f'Error: Could not download PDB {pdb_id} after multiple attempts')
    return None

# Download experimental AChE structure (with donepezil bound)
exp_pdb_id = '4EY7'
exp_pdb = download_pdb_experimental(exp_pdb_id)

if exp_pdb:
    print(f"Downloaded experimental structure: PDB {exp_pdb_id}")
    print("Human Acetylcholinesterase in complex with Donepezil")
    print("Method: X-ray crystallography, Resolution: 2.35 Å")
    print(f"File size: {len(exp_pdb):,} characters")

    # Count atoms
    atom_count = sum(1 for line in exp_pdb.split('\n') if line.startswith('ATOM'))
    hetatm_count = sum(1 for line in exp_pdb.split('\n') if line.startswith('HETATM'))
    print(f"Protein atoms: {atom_count:,}")
    print(f"Ligand/solvent atoms (HETATM): {hetatm_count:,}")


In [ ]:
# ============================================================
# VISUALIZE: EXPERIMENTAL STRUCTURE WITH BOUND DRUG
# ============================================================
# Show the experimental structure highlighting the drug (donepezil)
# in the active site of AChE.

if exp_pdb:
    view = py3Dmol.view(width=800, height=500)
    view.addModel(exp_pdb, 'pdb')
    
    # Show protein as cartoon (gray)
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightgray', 'opacity': 0.8}})
    
    # Highlight the drug molecule (HETATM, not water)
    # Donepezil has residue name 'E20' in this PDB
    view.setStyle({'resn': 'E20'}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.3}})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.5, 'color': 'green'}, {'resn': 'E20'})
    
    view.zoomTo({'resn': 'E20'})
    
    print("Experimental structure: AChE + Donepezil (green)")
    print("PDB: 4EY7 | The drug sits deep in the catalytic gorge of AChE.")
    print("Donepezil is used to treat mild-to-moderate Alzheimer's disease.")
    view.show()

In [ ]:
# ============================================================
# COMPARE ALPHAFOLD vs EXPERIMENTAL B-FACTORS
# ============================================================
# In experimental structures, B-factors (temperature factors) reflect
# atomic mobility/disorder. In AlphaFold, "B-factors" are pLDDT scores.
# Both correlate with structural order: ordered regions have low
# experimental B-factors and high AlphaFold pLDDT.

def extract_bfactors_chain(pdb_text, chain='A'):
    """Extract CA B-factors for a specific chain."""
    res_nums = []
    bfactors = []
    
    for line in pdb_text.split('\n'):
        if line.startswith('ATOM') and line[12:16].strip() == 'CA' and line[21] == chain:
            res_num = int(line[22:26].strip())
            b_factor = float(line[60:66].strip())
            res_nums.append(res_num)
            bfactors.append(b_factor)
    
    return np.array(res_nums), np.array(bfactors)

# Extract from experimental structure (chain A)
exp_res, exp_bfactors = extract_bfactors_chain(exp_pdb, chain='A')

# Compare range of residue numbers to find overlap
af_res, af_plddt, _ = extract_plddt_from_pdb(ache_pdb)

# Find common residues
common_res = np.intersect1d(exp_res, af_res)
print(f"Experimental structure residues (chain A): {len(exp_res)}")
print(f"AlphaFold prediction residues: {len(af_res)}")
print(f"Common residues: {len(common_res)}")

if len(common_res) > 50:
    # Get matching values
    exp_mask = np.isin(exp_res, common_res)
    af_mask = np.isin(af_res, common_res)
    
    exp_b_common = exp_bfactors[exp_mask]
    af_p_common = af_plddt[af_mask]
    
    # Plot comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: both along sequence
    ax1.plot(common_res, af_p_common, label='AlphaFold pLDDT', color='blue', alpha=0.7)
    ax1_twin = ax1.twinx()
    ax1_twin.plot(common_res, exp_b_common, label='Experimental B-factor', color='red', alpha=0.7)
    ax1.set_xlabel('Residue Number')
    ax1.set_ylabel('pLDDT (AlphaFold)', color='blue')
    ax1_twin.set_ylabel('B-factor (Experimental)', color='red')
    ax1.set_title('AlphaFold pLDDT vs Experimental B-factor')
    
    # Right: scatter plot (expect inverse correlation)
    ax2.scatter(exp_b_common, af_p_common, alpha=0.4, s=10)
    ax2.set_xlabel('Experimental B-factor (higher = more mobile)')
    ax2.set_ylabel('AlphaFold pLDDT (higher = more confident)')
    ax2.set_title('Correlation: Expect inverse relationship')
    
    # Add correlation coefficient
    corr = np.corrcoef(exp_b_common, af_p_common)[0, 1]
    ax2.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax2.transAxes, fontsize=12, va='top')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nPearson correlation between experimental B-factor and pLDDT: r = {corr:.3f}")
    print("Expected: negative correlation (high pLDDT → low B-factor → well-ordered)")
else:
    print("Insufficient overlap for comparison (numbering mismatch between structures).")
    print("This is common when comparing UniProt sequences to PDB structures.")

## Part 5: From Structure to Drug Design — The Complete Pipeline

Let's put it all together and understand how AlphaFold fits into the modern drug discovery pipeline:

```
Target Identification → AlphaFold Structure → Binding Site Analysis → 
Virtual Screening → Lead Optimization → Preclinical → Clinical Trials
```

### How AlphaFold Accelerates Drug Discovery

1. **No experimental structure needed**: Previously, if no X-ray/cryo-EM structure existed, structure-based drug design was impossible
2. **Coverage of the proteome**: AlphaFold provides structures for proteins that are difficult to crystallize (membrane proteins, disordered proteins)
3. **Speed**: Getting a predicted structure takes seconds vs months/years for experimental determination
4. **Novel targets**: Enables drug design for previously "undruggable" proteins

In [ ]:
# ============================================================
# DRUGGABILITY ASSESSMENT BASED ON STRUCTURAL FEATURES
# ============================================================
# A simple druggability score based on structural properties:
# 1. High mean pLDDT (well-structured = likely to have binding pockets)
# 2. Large structured regions (need a pocket to fit a drug)
# 3. Hydrophobic content (drug binding sites are often hydrophobic)

def assess_druggability(pdb_text, protein_name):
    """
    Simple druggability assessment based on AlphaFold prediction features.
    
    This is a simplified heuristic — real druggability assessment uses
    pocket detection algorithms (fpocket, SiteMap) and considers many
    more factors.
    """
    res_nums, plddt, res_names = extract_plddt_from_pdb(pdb_text)
    
    # Factor 1: Overall structural confidence
    mean_plddt = plddt.mean()
    structure_score = min(mean_plddt / 90.0, 1.0)  # Normalize to 0-1
    
    # Factor 2: Fraction of well-structured residues
    frac_structured = (plddt > 80).mean()
    
    # Factor 3: Hydrophobic content (drug pockets are hydrophobic)
    hydrophobic_aas = ['ALA', 'VAL', 'LEU', 'ILE', 'MET', 'PHE', 'TRP', 'PRO']
    frac_hydrophobic = sum(1 for r in res_names if r in hydrophobic_aas) / len(res_names)
    
    # Factor 4: Protein size (larger proteins more likely to have pockets)
    size_score = min(len(plddt) / 300.0, 1.0)  # Normalize
    
    # Combined druggability score (simple weighted average)
    druggability = (
        0.35 * structure_score +
        0.30 * frac_structured +
        0.20 * frac_hydrophobic * 3 +  # Scale up hydrophobic contribution
        0.15 * size_score
    )
    
    return {
        'protein': protein_name,
        'mean_plddt': mean_plddt,
        'frac_structured': frac_structured,
        'frac_hydrophobic': frac_hydrophobic,
        'size': len(plddt),
        'druggability_score': druggability
    }

# Assess all our targets
print("Simplified Druggability Assessment")
print("="*90)
print(f"{'Protein':<30} {'Size':<8} {'pLDDT':<8} {'Structured%':<13} {'Hydrophobic%':<14} {'Score':<8}")
print("-"*90)

# Include AChE as well
all_assessments = []
all_assessments.append(assess_druggability(ache_pdb, 'AChE'))

for uniprot_id, data in target_data.items():
    assessment = assess_druggability(data['pdb'], data['info']['name'])
    all_assessments.append(assessment)

for a in sorted(all_assessments, key=lambda x: x['druggability_score'], reverse=True):
    print(f"{a['protein']:<30} {a['size']:<8} {a['mean_plddt']:<8.1f} {100*a['frac_structured']:<13.1f} {100*a['frac_hydrophobic']:<14.1f} {a['druggability_score']:<8.3f}")

print("\nNote: This is a simplified heuristic for educational purposes.")
print("Real druggability assessment uses sophisticated pocket detection")
print("algorithms and considers binding site geometry, flexibility, and more.")
print("\nAll proteins above are known drug targets — they are all druggable!")
print("The scores help illustrate the structural features that contribute to druggability.")

In [ ]:
# ============================================================
# VISUALIZE DRUGGABILITY COMPARISON
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

proteins = [a['protein'] for a in all_assessments]
scores = [a['druggability_score'] for a in all_assessments]
plddts = [a['mean_plddt'] for a in all_assessments]
structured = [100*a['frac_structured'] for a in all_assessments]

# Bar chart: Druggability scores
colors = plt.cm.RdYlGn([s/max(scores) for s in scores])
axes[0].barh(proteins, scores, color=colors)
axes[0].set_xlabel('Druggability Score')
axes[0].set_title('Simplified Druggability Score')

# Bar chart: Mean pLDDT
axes[1].barh(proteins, plddts, color='steelblue')
axes[1].set_xlabel('Mean pLDDT')
axes[1].set_title('AlphaFold Confidence')
axes[1].set_xlim(60, 100)

# Bar chart: % well-structured
axes[2].barh(proteins, structured, color='seagreen')
axes[2].set_xlabel('% Residues with pLDDT > 80')
axes[2].set_title('Structural Coverage')

plt.tight_layout()
plt.show()

## Part 6: AlphaFold's Impact and Limitations

### Revolutionary Impact

AlphaFold has transformed structural biology and drug discovery:

- **CASP14 (2020)**: Achieved median GDT-TS of 92.4 — essentially solving the single-domain protein structure prediction problem
- **AlphaFold DB**: Released 200+ million predicted structures (July 2022)
- **Nobel Prize**: Demis Hassabis and John Jumper shared the 2024 Nobel Prize in Chemistry for AlphaFold

### Limitations to Keep in Mind

1. **Static structures only**: AlphaFold predicts one conformation, but proteins are dynamic — they move and flex
2. **No ligand binding**: AlphaFold doesn't predict how drugs bind (AlphaFold 3 starts to address this)
3. **Confidence varies**: Disordered regions and multi-domain proteins may have lower accuracy
4. **No post-translational modifications**: Glycosylation, phosphorylation etc. are not modeled
5. **Single chain focus**: Protein-protein interfaces and complexes are harder (AlphaFold-Multimer addresses this)

### AlphaFold 3 (2024)

AlphaFold 3 extends the capabilities to predict:
- Protein-ligand complexes (how drugs bind)
- Protein-DNA/RNA interactions
- Post-translational modifications
- Multi-chain complexes

Reference: Abramson et al. (2024). "Accurate structure prediction of biomolecular interactions with AlphaFold 3." *Nature*, 630:493-500.

In [ ]:
# ============================================================
# SUMMARY: WHAT WE LEARNED
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════════╗
║              DAY 4 PRACTICAL SUMMARY                           ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  Key Concepts Covered:                                         ║
║                                                                ║
║  1. The Protein Folding Problem                                ║
║     - Predicting 3D structure from amino acid sequence         ║
║     - 50+ year grand challenge in biology                      ║
║                                                                ║
║  2. AlphaFold Architecture                                     ║
║     - MSAs + Evoformer + Structure Module                      ║
║     - pLDDT confidence scores (0-100)                          ║
║                                                                ║
║  3. Accessing AlphaFold Predictions                            ║
║     - AlphaFold Database API                                   ║
║     - 200+ million protein structures available                ║
║                                                                ║
║  4. Structural Analysis for Drug Discovery                     ║
║     - Identifying druggable binding sites                      ║
║     - Comparing with experimental structures                   ║
║     - Understanding structural confidence                      ║
║                                                                ║
║  5. Impact on Drug Discovery                                   ║
║     - Structure-based drug design                              ║
║     - Enabling novel target investigation                      ║
║     - From days → seconds for structure prediction             ║
║                                                                ║
║  Tools Used:                                                   ║
║     - AlphaFold Database REST API                              ║
║     - py3Dmol for 3D visualization                             ║
║     - BioPython for structural parsing                         ║
║     - RCSB PDB for experimental structures                     ║
║                                                                ║
╚══════════════════════════════════════════════════════════════════╝

Next Steps (for further exploration):
  • Try predicting your own protein: https://colab.research.google.com/github/deepmind/alphafold/
  • Explore AlphaFold DB: https://alphafold.com/
  • Learn about molecular docking: AutoDock Vina, GNINA
  • Try AlphaFold 3 Server: https://alphafoldserver.com/
""")